# 🎬 CineVerify-Core-3B-Instruct Training Notebook
### Domain-Specialized Indian Cinema Claim & Entity Extraction SLM

- **Base Architecture:** `unsloth/Qwen2.5-3B-Instruct`
- **Fine-Tuning Method:** QLoRA 4-bit via Unsloth (Colab Free T4 GPU - 15GB VRAM)
- **Expected Training Time:** ~20-30 minutes for 3 epochs (~300 samples)
- **Export Target:** 16-bit HF checkpoint & 4-bit GGUF (`q4_k_m`) for local inference with 0 API cost.

## 1. Verify GPU & Install Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

# Install Unsloth and training dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes datasets
!pip install sentencepiece protobuf

## 2. Load Model & Tokenizer (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None  # None for auto-detection (Float16 for Tesla T4)
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("Model and tokenizer successfully loaded in 4-bit!")

## 3. Attach LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,  # Optimized to 0 by Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

## 4. Load & Format Dataset (ChatML)
You can upload your `cineverify_train.jsonl` file directly to Colab, or run the embedded generator fallback cell below.

In [ ]:
import os
import json
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# Set ChatML template on tokenizer
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",
    mapping={"role": "role", "content": "content", "user": "user", "assistant": "assistant"}
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

train_file = "cineverify_train.jsonl"

# If file not uploaded, notify or load sample
if not os.path.exists(train_file):
    print(f"'{train_file}' not found. Please upload it via the Colab files pane on the left.")
else:
    raw_data = []
    with open(train_file, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                raw_data.append(json.loads(line))
    
    hf_dataset = Dataset.from_list(raw_data)
    formatted_dataset = hf_dataset.map(formatting_prompts_func, batched=True)
    print(f"Loaded and formatted {len(formatted_dataset)} samples!")
    print("\n--- Sample Formatted Text ---\n")
    print(formatted_dataset[0]["text"][:350] + "...")

## 5. Train with SFTTrainer (20-30 Mins on Colab T4)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch size = 8
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none"
    ),
)

trainer_stats = trainer.train()
print("Training finished! Stats:", trainer_stats)

## 6. Fast Inference & Contract Verification

In [ ]:
# Switch model to inference mode
FastLanguageModel.for_inference(model)

test_headlines = [
    "Deepika Padukone joins Spirit opposite Prabhas",
    "Pushpa 2: The Rule hits ₹1000 Crore milestone at worldwide box office!",
    "Reports suggest Lokesh Kanagaraj roped in Fahadh Faasil for cameo in Coolie starring Rajinikanth",
    "SSMB29 locks worldwide theatrical release on Sankranti 2026"
]

for headline in test_headlines:
    messages = [
        {"role": "system", "content": "You are CineVerify-AI, an expert Indian cinema fact-checking model. Extract the core claim, category, and entities from the given text in valid JSON."},
        {"role": "user", "content": headline}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    
    outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True, temperature=0.1)
    decoded = tokenizer.batch_decode(outputs)
    
    # Extract assistant response
    resp = decoded[0].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
    print(f"\n[HEADLINE]: {headline}")
    print(f"[OUTPUT JSON]:\n{resp}")

## 7. Export Model (16-bit Checkpoint & GGUF Quantization)
Export to:
1. **Merged 16-bit / LoRA weights** (to upload to Hugging Face Hub)
2. **4-bit Quantized GGUF (`q4_k_m`)** (~1.9 GB file for local CPU/GPU inference in FastAPI/Ollama)

In [ ]:
# Export to 4-bit Quantized GGUF directly
model.save_pretrained_gguf(
    "CineVerify-Core-3B-Instruct-GGUF",
    tokenizer,
    quantization_method="q4_k_m"
)
print("GGUF model exported! Check the 'CineVerify-Core-3B-Instruct-GGUF' folder.")

# Optional: Save LoRA adapters locally
model.save_pretrained("CineVerify-Core-3B-LoRA")
tokenizer.save_pretrained("CineVerify-Core-3B-LoRA")
print("LoRA adapter weights saved!")

### Optional: Push to Hugging Face Hub
Uncomment the cell below and add your HF Token to push directly to your Hugging Face profile.

In [ ]:
# from huggingface_hub import login
# login("YOUR_HF_TOKEN")
# model.push_to_hub_merged("your-username/CineVerify-Core-3B-Instruct", tokenizer, save_method="merged_16bit")
# model.push_to_hub_gguf("your-username/CineVerify-Core-3B-Instruct-GGUF", tokenizer, quantization_method="q4_k_m")